# IMS Toolkit – End-to-End Guide

This notebook shows how to:
1. Register the IMS experiment as a dynamic toolkit.
2. Load it via `DataHandler_Class.getData`.
3. List IMS stations using the official API token.
4. Download a small time window for a single station (using the IMS code, unchanged).
5. Locate the saved files and load them to pandas.
6. Plot a simple *low-wind hourly average* chart.

> **Note:** You need an API token placed at `~/hera-ims/token.json` with a JSON key `Authorization`, e.g. `{ "Authorization": "ApiToken <YOUR_TOKEN>" }`.


In [ ]:

PROJECT = "UnitTestProject"
IMS_ROOT = os.path.expanduser("~/hera-ims")
IMS_CODE = os.path.join(IMS_ROOT, "code")
IMS_DATA = os.path.join(IMS_ROOT, "data")
os.makedirs(IMS_DATA, exist_ok=True)
print("Project:", PROJECT)
print("IMS code dir:", IMS_CODE)
print("IMS data dir:", IMS_DATA)


## 1) Register IMS (two options)
### Option A – CLI

In [ ]:

        print(
            "python -m hera.utils.data.cli_toolkit_repository register-datasource \n"
            f"  --project {PROJECT} \
"
            "  --name IMS \
"
            "  --classpath IMS_experiment.IMS_experiment \
"
            f"  --resource {IMS_CODE} \
"
            f"  --params '{{"projectName":"{PROJECT}","pathToExperiment":"{IMS_ROOT}","filesDirectory":"{IMS_DATA}"}}' \
"
            "  --version 0.0.1 \
"
            "  --overwrite"
        )


### Option B – Python API

In [ ]:

import sys, importlib, os
from hera.toolkit import ToolkitHome

# Ensure IMS code folder is importable for locating the class:
if IMS_CODE not in sys.path:
    sys.path.insert(0, IMS_CODE)

IMSClass = getattr(importlib.import_module("IMS_experiment"), "IMS_experiment")

th = ToolkitHome()
doc = th.registerToolkit(
    toolkitclass=IMSClass,
    projectName=PROJECT,
    datasource_name="IMS",
    params={"projectName": PROJECT, "pathToExperiment": IMS_ROOT, "filesDirectory": IMS_DATA},
    version=(0, 0, 1),
    overwrite=True,
)
print("Registered IMS as ToolkitDataSource. Resource:", doc.resource)


## 2) Confirm registration and discover toolkits

In [ ]:

from hera.toolkit import ToolkitHome
th = ToolkitHome()
df = th.getToolkitTable(PROJECT)
df


## 3) Load the IMS object via DataHandler_Class

In [ ]:

from hera.datalayer import Project
from hera.datalayer.datahandler import DataHandler_Class

p = Project(projectName=PROJECT)
docs = p.getMeasurementsDocuments(type="ToolkitDataSource", datasourceName="IMS")
assert docs, "IMS datasource not found. Did you register it above?"
doc = docs[0]

ims = DataHandler_Class.getData(resource=doc.resource, desc=doc.desc)
print("Loaded:", ims, "| filesDirectory:", getattr(ims, "filesDirectory", None))


## 4) List stations from the official IMS API

In [ ]:

import json, requests, os, pandas as pd

token_path = os.path.expanduser("~/hera-ims/token.json")
if not os.path.exists(token_path):
    print("Missing token file:", token_path)
    print("Expected JSON: { \"Authorization\": \"ApiToken <YOUR_TOKEN>\" }")
    stations_df = None
else:
    tok = json.load(open(token_path))
    url = "https://api.ims.gov.il/v1/Envista/stations"
    r = requests.get(url, headers={"Authorization": tok["Authorization"]}, timeout=30)
    print("HTTP:", r.status_code)
    data = r.json()
    stations_df = pd.json_normalize(data)
    print("Stations:", len(stations_df))
    display(stations_df.head(10))


## 5) Download a small window for a single station
The IMS code exposes `download(station='', start='', end='latest')`.

- `station` – a **name** (e.g., *'BET ZAYDA'*). If empty, it may fetch all.
- `start` – `'YYYY-MM-DD'` or empty for station open date.
- `end` – `'YYYY-MM-DD'` or `'latest'` (≈ up to ~3 months back).


In [ ]:

import time
target_station = "BET ZAYDA"  # change as you like
start_date = ""               # e.g. "2024-01-01" or ""
end_date   = "latest"         # or "2024-03-31"

t0 = time.time()
try:
    ims.download(station=target_station, start=start_date, end=end_date)
    print("Download invoked for:", target_station)
except Exception as e:
    print("⚠️ download failed:", e)
finally:
    print("elapsed: %.2fs" % (time.time()-t0))


## 6) Locate files saved by IMS and preview

In [ ]:

import glob, os, pandas as pd
out_dir = getattr(ims, "filesDirectory", IMS_DATA)
files = sorted(glob.glob(os.path.join(out_dir, "**", "*.parquet"), recursive=True) +
               glob.glob(os.path.join(out_dir, "**", "*.csv"), recursive=True))
print("Found files:", len(files))
for f in files[:5]:
    print("-", f)

if files:
    f = files[0]
    try:
        df = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)
        display(df.head(10))
    except Exception as e:
        print("Preview failed for", f, "->", e)


## 7) Plot a *low-wind hourly average* chart
Below is a generic helper function. It tries to:
- Identify a datetime column, parse to pandas datetime.
- Auto-detect a wind-speed column (`WS`, `Wind Speed`, etc.).
- Filter rows with speed ≤ `ws_threshold` (default 2 m/s), group by hour-of-day, and plot mean.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

def plot_low_wind_hourly_average(df, ws_threshold=2.0):
    if df is None or df.empty:
        print("No data provided.")
        return

    # Guess time column
    time_cols = [c for c in df.columns if c.lower() in ("time","datetime","date","timestamp")]
    if not time_cols:
        # looser heuristic
        time_cols = [c for c in df.columns if "time" in c.lower() or "date" in c.lower()]
    if not time_cols:
        raise ValueError("Could not locate a time/datetime column.")

    tcol = time_cols[0]
    dt = pd.to_datetime(df[tcol], errors="coerce", utc=True).dropna()
    if dt.empty:
        raise ValueError("Datetime parsing failed.")

    # Guess wind-speed column
    candidates = [c for c in df.columns if c.upper() in ("WS","WS1MM","WS10MM")]
    if not candidates:
        # more relaxed
        candidates = [c for c in df.columns if "ws" in c.lower() or "wind" in c.lower()]
    if not candidates:
        raise ValueError("Could not find a wind-speed column (e.g., 'WS').")

    wcol = candidates[0]
    tmp = pd.DataFrame({ "dt": dt, "ws": pd.to_numeric(df[wcol], errors="coerce") }).dropna()
    if tmp.empty:
        raise ValueError("No numeric wind-speed samples.")

    low = tmp[tmp["ws"] <= ws_threshold].copy()
    if low.empty:
        print("No samples under threshold", ws_threshold)
        return

    low["hour"] = low["dt"].dt.tz_convert(None).dt.hour
    mean_by_hour = low.groupby("hour")["ws"].mean()

    plt.figure()
    mean_by_hour.plot(marker="o")
    plt.title(f"Low-wind hourly mean (≤ {ws_threshold} m/s)")
    plt.xlabel("Hour of day")
    plt.ylabel("Mean wind speed (m/s)")
    plt.grid(True)
    plt.show()

# Example usage on the previewed df (if available):
try:
    if 'df' in globals():
        plot_low_wind_hourly_average(df, ws_threshold=2.0)
except Exception as e:
    print("Plot failed:", e)


---
**Done.** Use these cells as a recipe: register → load → download → explore → plot.